# פרק 7: בניית אפליקציות צ'אט
## התחלה מהירה עם OpenAI API

פנקס רשימות זה מותאם מ-[מאגר דוגמאות Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) הכולל פנקסי רשימות שניגשים לשירותי [Azure OpenAI](notebook-azure-openai.ipynb).

ממשק ה-API של OpenAI בפייתון עובד גם עם דגמי Azure OpenAI, עם כמה שינויים. למדו עוד על ההבדלים כאן: [כיצד לעבור בין נקודות קצה של OpenAI ו-Azure OpenAI באמצעות Python](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# סקירה  
"מודלים לשוניים גדולים הם פונקציות שממפות טקסט לטקסט. בהינתן מחרוזת טקסט קלט, מודל לשוני גדול מנסה לחזות את הטקסט שיבוא בהמשך"(1). מחברת ה"קוויקסטארט" הזו תציג למשתמשים מושגים ברמת גובה של LLM, דרישות חבילה עיקריות להתחלות עם AML, הקדמה רכה לעיצוב פרומפט, וכמה דוגמאות קצרות לשימושים שונים. 


## תוכן העניינים  

[סקירה כללית](#overview)  
[כיצד להשתמש בשירות OpenAI](#how-to-use-openai-service)  
[1. יצירת שירות OpenAI שלך](#1.-creating-your-openai-service)  
[2. התקנה](#2.-installation)    
[3. האישורים](#3.-credentials)  

[מקרי שימוש](#use-cases)    
[1. סיכום טקסט](#1.-summarize-text)  
[2. סיווג טקסט](#2.-classify-text)  
[3. יצירת שמות חדשים למוצרים](#3.-generate-new-product-names)  
[4. כוונון מדויק של מסווג](#4.fine-tune-a-classifier)  

[הפניות](#references)


### בנה את הפקודה הראשונה שלך  
תרגיל קצר זה יספק מבוא בסיסי להגשת פקודות למודל של OpenAI למשימה פשוטה של "סיכום".  


**שלבים**:  
1. התקן את ספריית OpenAI בסביבת הפייתון שלך  
2. טען ספריות עזר סטנדרטיות והגדר את אישורי האבטחה הרגילים שלך לשירות OpenAI שיצרת  
3. בחר מודל למשימה שלך  
4. צור פקודה פשוטה עבור המודל  
5. שלח את הבקשה שלך ל-API של המודל!  


### 1. התקן את OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. ייבא ספריות עזר ואתחל הרשאות


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. מציאת הדגם המתאים  
דגמים כמו GPT-4o ו-GPT-4o mini יכולים להבין וליצור שפה טבעית, וזמינים בפלטפורמת OpenAI עם רמות שונות של עוצמה ומהירות המתאימות למשימות שונות.


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. עיצוב פרומפט  

"הקסם של מודלים גדולים לשפה הוא שכאשר הם מאומנים למזער את שגיאת החיזוי הזו בכמויות עצומות של טקסט, המודלים לומדים בסופו של דבר מושגים שימושיים לחיזויים אלו. למשל, הם לומדים מושגים כמו"(1):

* איך לאיית
* איך עובדת הדקדוק
* איך לעשות פרפרזה
* איך לענות על שאלות
* איך לנהל שיחה
* איך לכתוב בשפות רבות
* איך לתכנת
* וכו'.

#### איך לשלוט במודל שפה גדול  
"מבין כל הקלטים למודל שפה גדול, הפרומפט הטקסטואלי הוא ללא ספק המשפיע ביותר(1).

ניתן לגרום למודלים גדולים לשפה להפיק פלט בכמה דרכים:

הוראה: אמור למודל מה אתה רוצה
השלמה: גרום למודל להשלים את תחילת מה שאתה רוצה
הדגמה: הראה למודל מה אתה רוצה, באמצעות:
כמה דוגמאות בפרומפט
מאות או אלפי דוגמאות במערך אימון לכיוונון מדויק"



#### יש שלושה כללים בסיסיים ליצירת פרומפטים:

**הראה וספר**. הבהר מה אתה רוצה באמצעות הוראות, דוגמאות או שילוב של השניים. אם אתה רוצה שהמודל ידרג רשימת פריטים בסדר אלפביתי או יסווג פסקה לפי סנטימנט, הראה לו שזה מה שאתה רוצה.

**ספק נתונים באיכות גבוהה**. אם אתה מנסה לבנות ממיין או לגרום למודל לעקוב אחר דפוס, ודא שיש מספיק דוגמאות. הקפד לקרוא שוב את הדוגמאות שלך — בדרך כלל המודל חכם מספיק לראות טעויות איות בסיסיות ולתת תשובה, אבל הוא עלול גם להניח שזה מכוון וזה עשוי להשפיע על התגובה.

**בדוק את ההגדרות שלך.** ההגדרות temperature ו-top_p שולטות כמה דטרמיניסטי המודל ביצירת התגובה. אם אתה מבקש תגובה שיש לה תשובה נכונה אחת בלבד, רצוי להוריד אותן. אם אתה מחפש תגובות מגוונות יותר, אולי כדאי להעלות אותן. הטעות מספר אחת שהאנשים עושים עם הגדרות אלו היא להניח שהן שולטות ב"חכמה" או "יצירתיות".


מקור: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. הגש!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### חזור על אותו קריאה, איך התוצאות משתוות? 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## לסכם טקסט  
#### אתגר  
לסכם טקסט על ידי הוספת 'tl;dr:' לסוף קטע טקסט. שימו לב כיצד המודל מבין כיצד לבצע מספר משימות ללא הוראות נוספות. ניתן להתנסות עם ניסוחים מתארים יותר מ-tl;dr כדי לשנות את התנהגות המודל ולהתאים אישית את הסיכום שתקבלו(3).  

עבודה עדכנית הראתה שיפורים משמעותיים בהרבה משימות ומדדים בתחום עיבוד השפה הטבעית על ידי אימון מוקדם על קורפוס גדול של טקסט ואחריו כוונון נוסף למשימה ספציפית. למרות שהארכיטקטורה בדרך כלל אינה תלויה במשימה, שיטה זו עדיין דורשת מערכי נתונים לכוונון נוסף למשימות ספציפיות עם אלפי או עשרות אלפי דוגמאות. לעומת זאת, בני אדם בדרך כלל יכולים לבצע משימת שפה חדשה עם רק כמה דוגמאות או הוראות פשוטות - משהו שמערכות NLP עכשוויות עדיין מתקשות לבצע במידה רבה. כאן אנו מראים שהגדלת גודל מודלי השפה משפרת מאוד את הביצועים במשימות שונות עם מספר מוגבל של דוגמאות, לפעמים אפילו מגיעה לתחרותיות עם גישות הכוונון המתקדמות הקודמות.  



tl;dr  


# תרגילים למספר מקרי שימוש  
1. סיכום טקסט  
2. סיווג טקסט  
3. יצירת שמות למוצרים חדשים


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## סיווג טקסט  
#### אתגר  
סווג פריטים לקטגוריות המסופקות בזמן הביצוע. בדוגמה הבאה, אנו מספקים גם את הקטגוריות וגם את הטקסט לסיווג בהנחיה (*playground_reference). 

פנייה מלקוח: שלום, אחד מהמפתחות במקלדת הנייד שלי נשבר לאחרונה ואצטרך תחליף:

קטגוריה מסווגת:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## צור שמות מוצרים חדשים
#### אתגר
צור שמות מוצרים מתוך מילים לדוגמה. כאן אנו כוללים בהנחיה מידע על המוצר שאנו עומדים ליצור לו שמות. אנו גם מספקים דוגמה דומה להראות את הדפוס שאנו מעוניינים לקבל. הגדרנו גם ערך טמפרטורה גבוה כדי להגדיל את האקראיות ואת התגובות החדשניות יותר.

תיאור מוצר: יצרן שייקים ביתי
מילים בסיס: מהיר, בריא, קומפקטי.
שמות מוצרים: HomeShaker, Fit Shaker, QuickShake, Shake Maker

תיאור מוצר: זוג נעליים שניתן להתאים לכל מידה של רגל.
מילים בסיס: ניתן להתאמה, מתאים, מתאים-לכל.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# הפניות  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Best practices for fine-tuning GPT models to classify text](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# לעזרה נוספת  
[צוות המסחור של OpenAI](AzureOpenAITeam@microsoft.com) 


# תורמים
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
